# Multimodal Fashion & Context Retrieval — Demo

Thin demo on top of the `indexer/` and `retriever/` modules. Runs on a free Colab **T4 GPU** (Runtime -> Change runtime type -> T4 GPU).

Pipeline: FashionCLIP global + garment-region embeddings -> Chroma -> query decomposition -> compositional AND-scoring -> optional VQA re-rank.

## 1. Setup

In [ ]:
# If running in Colab, clone your repo (replace with your GitHub URL):
# !git clone https://github.com/<you>/fashion-retrieval.git
# %cd fashion-retrieval
!pip -q install -r requirements.txt

## 2. Fetch a Fashionpedia subset (~800 images)

In [ ]:
!python -m eval.fetch_data --n 800

## 3. Build the index (Part A)
Global scene vector + per-garment region vectors -> Chroma.

In [ ]:
!python -m indexer.build --data data/images

## 4. Run the 5 evaluation queries (Part B)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from retriever.search import FashionRetriever
from retriever.rerank import VQAReranker
from retriever.query_parser import parse

retriever = FashionRetriever()
reranker = VQAReranker()  # local BLIP-VQA, no API key

def show(query, k=5, rerank=True):
    res = retriever.search(query, k=k*4 if rerank else k)
    if rerank:
        res = reranker.rerank(parse(query), res)[:k]
    else:
        res = res[:k]
    fig, axes = plt.subplots(1, k, figsize=(4*k, 4))
    for ax, r in zip(axes, res):
        ax.imshow(Image.open(r.path)); ax.axis('off')
        ax.set_title(f"{r.score:.2f}", fontsize=10)
    fig.suptitle(query, fontsize=13); plt.tight_layout(); plt.show()

In [ ]:
queries = [
    'A person in a bright yellow raincoat.',
    'Professional business attire inside a modern office.',
    'Someone wearing a blue shirt sitting on a park bench.',
    'Casual weekend outfit for a city walk.',
    'A red tie and a white shirt in a formal setting.',
]
for q in queries:
    show(q, k=5, rerank=True)

## 5. Evaluation — beat vanilla CLIP
Colour-swap binding test + global-only vs full-pipeline ablation.

In [ ]:
!python -m eval.evaluate